In [ ]:

import os

# Each parallel worker should not itself use many BLAS threads.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")


# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from pmdarima import auto_arima

from joblib import Parallel, delayed


In [ ]:
# ============================================================
# 2. SETTINGS
# ============================================================

DATA_FILE = "//"

# For the 1000-client random pool:
N_SELECTED_CLIENTS = 100
POOL_NAME = "1000"



HISTORY = 100
HORIZON = 28

TRAIN_RATIO = 0.60
CAL_RATIO = 0.20

ALPHA = 0.10


# ------------------------------------------------------------
# AutoARIMA settings
# ------------------------------------------------------------

SEASONAL = False
M = 1

START_P = 0
START_Q = 0

# Smaller search space for speed
MAX_P = 3
MAX_Q = 3
MAX_D = 2


# ------------------------------------------------------------
# Parallelization
# ------------------------------------------------------------

# Start with 4 on Mac.
# If your laptop becomes too hot / slow, change to 2.
N_JOBS = 4


# ============================================================
# 3. LOAD DATA
# ============================================================

data = pd.read_parquet(
    DATA_FILE
)

print(
    "Data shape:",
    data.shape
)

print(
    "Number of clients in pool:",
    data["eup_grid_id"].nunique()
)


# ============================================================
# 4. SELECT HIGH-VARIABILITY CLIENTS
# ============================================================

client_stats = (
    data
    .groupby(
        "eup_grid_id"
    )[
        "balance_in_euro"
    ]
    .agg(
        mean="mean",
        std="std"
    )
)


selected_clients = (
    client_stats
    .sort_values(
        "std",
        ascending=False
    )
    .head(
        N_SELECTED_CLIENTS
    )
    .index
    .tolist()
)


print(
    "\nSelected clients:",
    len(selected_clients)
)

print(
    selected_clients
)


# Save IDs so DeepAR and NN can use exactly the same clients
selected_clients_file = (
    f"selected_clients_pool_{POOL_NAME}"
    f"_top_{N_SELECTED_CLIENTS}.csv"
)

pd.DataFrame(
    {
        "eup_grid_id":
            selected_clients
    }
).to_csv(
    selected_clients_file,
    index=False
)

print(
    "Selected client IDs saved to:",
    selected_clients_file
)


# ============================================================
# 5. WINKLER SCORE
# ============================================================

def winkler_score(
    y_true,
    lower,
    upper,
    alpha=0.10
):

    y_true = (
        np.asarray(
            y_true
        )
        .flatten()
    )

    lower = (
        np.asarray(
            lower
        )
        .flatten()
    )

    upper = (
        np.asarray(
            upper
        )
        .flatten()
    )

    width = (
        upper
        - lower
    )

    scores = width.copy()

    below = (
        y_true
        < lower
    )

    above = (
        y_true
        > upper
    )


    scores[
        below
    ] += (
        2 / alpha
    ) * (
        lower[below]
        - y_true[below]
    )


    scores[
        above
    ] += (
        2 / alpha
    ) * (
        y_true[above]
        - upper[above]
    )


    return np.mean(
        scores
    )


# ============================================================
# 6. CREATE SLIDING WINDOWS
# ============================================================

def make_windows(
    y,
    history=100,
    horizon=28
):

    X = []
    Y = []

    for i in range(
        len(y)
        - history
        - horizon
        + 1
    ):

        X.append(
            y[
                i:
                i + history
            ]
        )

        Y.append(
            y[
                i + history:
                i + history + horizon
            ]
        )

    return (
        np.asarray(X),
        np.asarray(Y)
    )


# ============================================================
# 7. FIT ONE AUTOARIMA FORECAST ORIGIN
# ============================================================

def fit_autoarima_origin(
    history_window,
    horizon=28
):

    model = auto_arima(

        history_window,

        start_p=START_P,
        start_q=START_Q,

        max_p=MAX_P,
        max_q=MAX_Q,
        max_d=MAX_D,

        seasonal=SEASONAL,
        m=M,

        information_criterion="aic",

        stepwise=True,

        suppress_warnings=True,

        error_action="ignore",

        trace=False
    )


    forecast, conf_int = (
        model.predict(
            n_periods=horizon,
            return_conf_int=True,
            alpha=ALPHA
        )
    )


    lower = (
        conf_int[
            :, 0
        ]
    )

    upper = (
        conf_int[
            :, 1
        ]
    )


    return (
        np.asarray(
            forecast
        ),

        np.asarray(
            lower
        ),

        np.asarray(
            upper
        ),

        model.order
    )


# ============================================================
# 8. FORECAST MULTIPLE ORIGINS
# ============================================================

def forecast_windows_autoarima(
    X,
    horizon=28
):

    forecasts = []
    lowers = []
    uppers = []
    orders = []


    for history_window in X:

        try:

            (
                forecast,
                lower,
                upper,
                order

            ) = fit_autoarima_origin(

                history_window,
                horizon=horizon
            )


        except Exception:

            forecast = np.full(
                horizon,
                np.nan
            )

            lower = np.full(
                horizon,
                np.nan
            )

            upper = np.full(
                horizon,
                np.nan
            )

            order = None


        forecasts.append(
            forecast
        )

        lowers.append(
            lower
        )

        uppers.append(
            upper
        )

        orders.append(
            order
        )


    return (
        np.asarray(
            forecasts
        ),

        np.asarray(
            lowers
        ),

        np.asarray(
            uppers
        ),

        orders
    )


# ============================================================
# 9. SINGLE-CLIENT AUTOARIMA PIPELINE
# ============================================================

def run_autoarima_pipeline(
    series,
    dates
):

    series = np.asarray(
        series,
        dtype=float
    )

    dates = pd.to_datetime(
        dates
    )


    # ========================================================
    # LOG1P CHECK
    # ========================================================

    if np.any(
        series <= -1
    ):

        raise ValueError(
            "balance_in_euro contains values <= -1. "
            "np.log1p() cannot be applied safely."
        )


    # ========================================================
    # LOG TRANSFORMATION
    # ========================================================

    y_log = np.log1p(
        series
    )


    # ========================================================
    # SLIDING WINDOWS
    # ========================================================

    X, Y = make_windows(

        y_log,

        history=HISTORY,

        horizon=HORIZON
    )


    if len(X) < 50:

        return None


    # ========================================================
    # CHRONOLOGICAL 60 / 20 / 20 SPLIT
    # ========================================================

    n = len(X)


    train_end = int(
        n
        * TRAIN_RATIO
    )


    cal_end = int(
        n
        * (
            TRAIN_RATIO
            + CAL_RATIO
        )
    )


    X_train = (
        X[
            :train_end
        ]
    )

    Y_train = (
        Y[
            :train_end
        ]
    )


    X_cal = (
        X[
            train_end:
            cal_end
        ]
    )

    Y_cal = (
        Y[
            train_end:
            cal_end
        ]
    )


    X_test = (
        X[
            cal_end:
        ]
    )

    Y_test = (
        Y[
            cal_end:
        ]
    )


    if (
        len(X_cal) == 0
        or len(X_test) == 0
    ):

        return None


    # ========================================================
    # CALIBRATION FORECASTS
    # ========================================================

    (
        forecast_cal,
        lower_cal,
        upper_cal,
        cal_orders

    ) = forecast_windows_autoarima(

        X_cal,
        horizon=HORIZON
    )


    # --------------------------------------------------------
    # Valid calibration windows
    # --------------------------------------------------------

    valid_cal = (

        ~np.isnan(
            forecast_cal
        ).any(
            axis=1
        )

        &

        ~np.isnan(
            lower_cal
        ).any(
            axis=1
        )

        &

        ~np.isnan(
            upper_cal
        ).any(
            axis=1
        )
    )


    forecast_cal = (
        forecast_cal[
            valid_cal
        ]
    )

    lower_cal = (
        lower_cal[
            valid_cal
        ]
    )

    upper_cal = (
        upper_cal[
            valid_cal
        ]
    )

    Y_cal_valid = (
        Y_cal[
            valid_cal
        ]
    )


    if len(
        Y_cal_valid
    ) == 0:

        return None


    # ========================================================
    # HORIZON-SPECIFIC CONFORMAL SCORES
    # ========================================================

    scores = np.maximum(

        lower_cal
        - Y_cal_valid,

        Y_cal_valid
        - upper_cal
    )


    qhat = []


    for h in range(
        HORIZON
    ):

        q_h = np.quantile(

            scores[
                :, h
            ],

            1 - ALPHA,

            method="higher"
        )


        # Conformal may widen or leave unchanged,
        # but does not shrink the original PI.
        qhat.append(

            max(
                0.0,
                float(q_h)
            )
        )


    qhat = np.asarray(
        qhat
    )


    # ========================================================
    # TEST FORECASTS
    # ========================================================

    (
        forecast_test,
        lower_test,
        upper_test,
        test_orders

    ) = forecast_windows_autoarima(

        X_test,
        horizon=HORIZON
    )


    valid_test = (

        ~np.isnan(
            forecast_test
        ).any(
            axis=1
        )

        &

        ~np.isnan(
            lower_test
        ).any(
            axis=1
        )

        &

        ~np.isnan(
            upper_test
        ).any(
            axis=1
        )
    )


    # Keep global window index for correct plotting
    test_global_indices = np.arange(
        cal_end,
        n
    )

    valid_test_global_indices = (
        test_global_indices[
            valid_test
        ]
    )


    forecast_test = (
        forecast_test[
            valid_test
        ]
    )

    lower_test = (
        lower_test[
            valid_test
        ]
    )

    upper_test = (
        upper_test[
            valid_test
        ]
    )

    Y_test_valid = (
        Y_test[
            valid_test
        ]
    )


    if len(
        Y_test_valid
    ) == 0:

        return None


    # ========================================================
    # CONFORMAL TEST INTERVALS
    # LOG SCALE
    # ========================================================

    lower_conf_log = (

        lower_test

        - qhat[
            None, :
        ]
    )


    upper_conf_log = (

        upper_test

        + qhat[
            None, :
        ]
    )


    # ========================================================
    # BACK TO EUR
    # ========================================================

    actual = np.expm1(
        Y_test_valid
    )


    forecast = np.expm1(
        forecast_test
    )


    lower_conf = np.expm1(
        lower_conf_log
    )


    upper_conf = np.expm1(
        upper_conf_log
    )


    # ========================================================
    # METRICS
    # ========================================================

    rmse = np.sqrt(

        mean_squared_error(

            actual.flatten(),

            forecast.flatten()
        )
    )


    coverage = np.mean(

        (
            actual
            >= lower_conf
        )

        &

        (
            actual
            <= upper_conf
        )
    )


    winkler = winkler_score(

        actual,

        lower_conf,

        upper_conf,

        alpha=ALPHA
    )


    # --------------------------------------------------------
    # Normalization
    # Keep consistent with DeepAR and NN
    # --------------------------------------------------------

    std_balance = (

        np.std(
            series
        )

        + 1e-8
    )


    nrmse = (

        rmse
        / std_balance
    )


    nwinkler = (

        winkler
        / std_balance
    )


    # ========================================================
    # LAST VALID TEST WINDOW FOR PLOTTING
    # ========================================================

    last_global_window_index = (

        valid_test_global_indices[
            -1
        ]
    )


    forecast_start = (

        last_global_window_index
        + HISTORY
    )


    history_start = (

        forecast_start
        - HISTORY
    )


    history_dates = (

        dates[
            history_start:
            forecast_start
        ]
    )


    history_values = (

        series[
            history_start:
            forecast_start
        ]
    )


    future_dates = (

        dates[
            forecast_start:
            forecast_start
            + HORIZON
        ]
    )


    return {

        "rmse":
            rmse,

        "coverage":
            coverage,

        "winkler":
            winkler,

        "nrmse":
            nrmse,

        "nwinkler":
            nwinkler,

        "std_balance":
            std_balance,

        "actual":
            actual,

        "forecast":
            forecast,

        "lower_conf":
            lower_conf,

        "upper_conf":
            upper_conf,

        "qhat":
            qhat,

        "series":
            series,

        "dates":
            dates,

        "history_dates":
            history_dates,

        "history_values":
            history_values,

        "future_dates":
            future_dates,

        "n_train_windows":
            len(X_train),

        "n_cal_windows":
            len(Y_cal_valid),

        "n_test_windows":
            len(Y_test_valid),

        "cal_orders":
            cal_orders,

        "test_orders":
            test_orders
    }


# ============================================================
# 10. PROCESS ONE CLIENT
# ============================================================

def process_client(
    client_id
):

    try:

        subset = (
            data[
                data[
                    "eup_grid_id"
                ]
                == client_id
            ]
            .copy()
        )


        subset[
            "date"
        ] = pd.to_datetime(

            subset[
                "date"
            ]
        )


        subset = (

            subset
            .sort_values(
                "date"
            )
            .set_index(
                "date"
            )
            .asfreq(
                "D"
            )
        )


        subset[
            "balance_in_euro"
        ] = (

            subset[
                "balance_in_euro"
            ]
            .ffill()
            .fillna(0)
        )


        series = (

            subset[
                "balance_in_euro"
            ]
            .astype(float)
            .values
        )


        result = run_autoarima_pipeline(

            series,

            subset.index
        )


        if result is None:

            return None


        metrics = {

            "client_id":
                client_id,

            "rmse":
                result[
                    "rmse"
                ],

            "coverage":
                result[
                    "coverage"
                ],

            "winkler":
                result[
                    "winkler"
                ],

            "nrmse":
                result[
                    "nrmse"
                ],

            "nwinkler":
                result[
                    "nwinkler"
                ],

            "std_balance":
                result[
                    "std_balance"
                ],

            "n_train_windows":
                result[
                    "n_train_windows"
                ],

            "n_cal_windows":
                result[
                    "n_cal_windows"
                ],

            "n_test_windows":
                result[
                    "n_test_windows"
                ]
        }


        return (
            client_id,
            result,
            metrics
        )


    except Exception as e:

        print(
            f"FAILED {client_id}: {e}"
        )

        return None


# ============================================================
# 11. RUN CLIENTS IN PARALLEL
# ============================================================

print(
    "\n========================================"
)

print(
    "STARTING PARALLEL AUTOARIMA"
)

print(
    "========================================"
)

print(
    "Number of clients:",
    len(selected_clients)
)

print(
    "Parallel jobs:",
    N_JOBS
)

print(
    "AutoARIMA search:",
    f"p <= {MAX_P}, "
    f"q <= {MAX_Q}, "
    f"d <= {MAX_D}"
)


processed = Parallel(

    n_jobs=N_JOBS,

    backend="loky",

    verbose=10

)(
    delayed(
        process_client
    )(
        client_id
    )

    for client_id
    in selected_clients
)


# ============================================================
# 12. COLLECT RESULTS
# ============================================================

all_results = {}
all_metrics = []


for item in processed:

    if item is None:

        continue


    (
        client_id,
        result,
        metrics
    ) = item


    all_results[
        client_id
    ] = result


    all_metrics.append(
        metrics
    )


    print(

        f"Done: {client_id} | "

        f"NRMSE="
        f"{result['nrmse']:.3f} | "

        f"Coverage="
        f"{result['coverage']:.2%} | "

        f"NWinkler="
        f"{result['nwinkler']:.3f}"
    )


# ============================================================
# 13. CLIENT-LEVEL RESULTS
# ============================================================

metrics_df = pd.DataFrame(
    all_metrics
)


if len(
    metrics_df
) == 0:

    raise RuntimeError(
        "No clients were successfully evaluated."
    )


print(
    "\nNumber of successfully evaluated clients:",
    len(metrics_df)
)


print(
    metrics_df[
        [
            "client_id",
            "nrmse",
            "coverage",
            "nwinkler"
        ]
    ]
)


# ============================================================
# 14. AGGREGATE RESULTS
# ============================================================

aggregate_metrics = {

    "n_clients":
        len(
            metrics_df
        ),

    "mean_nrmse":
        metrics_df[
            "nrmse"
        ].mean(),

    "median_nrmse":
        metrics_df[
            "nrmse"
        ].median(),

    "mean_coverage":
        metrics_df[
            "coverage"
        ].mean(),

    "median_coverage":
        metrics_df[
            "coverage"
        ].median(),

    "mean_nwinkler":
        metrics_df[
            "nwinkler"
        ].mean(),

    "median_nwinkler":
        metrics_df[
            "nwinkler"
        ].median()
}


print(
    "\n========================================"
)

print(
    "AGGREGATE CONFORMAL AUTOARIMA RESULTS"
)

print(
    "========================================"
)


print(
    f"Number of clients: "
    f"{aggregate_metrics['n_clients']}"
)


print(
    f"Mean NRMSE: "
    f"{aggregate_metrics['mean_nrmse']:.3f}"
)


print(
    f"Median NRMSE: "
    f"{aggregate_metrics['median_nrmse']:.3f}"
)


print(
    f"Mean Coverage: "
    f"{aggregate_metrics['mean_coverage']:.2%}"
)


print(
    f"Median Coverage: "
    f"{aggregate_metrics['median_coverage']:.2%}"
)


print(
    f"Mean Normalized Winkler: "
    f"{aggregate_metrics['mean_nwinkler']:.3f}"
)


print(
    f"Median Normalized Winkler: "
    f"{aggregate_metrics['median_nwinkler']:.3f}"
)


# ============================================================
# 15. PRIMARY THESIS RESULTS
# ============================================================

print(
    "\nPRIMARY THESIS METRICS"
)

print(
    "----------------------"
)

print(
    f"Median NRMSE: "
    f"{aggregate_metrics['median_nrmse']:.3f}"
)

print(
    f"Mean Coverage: "
    f"{aggregate_metrics['mean_coverage']:.2%}"
)

print(
    f"Median Normalized Winkler: "
    f"{aggregate_metrics['median_nwinkler']:.3f}"
)


# ============================================================
# 16. LOWER / HIGHER ERROR CLIENTS
# ============================================================

best_3 = (

    metrics_df

    .sort_values(
        "nrmse",
        ascending=True
    )

    .head(3)
)


worst_3 = (

    metrics_df

    .sort_values(
        "nrmse",
        ascending=False
    )

    .head(3)
)


print(
    "\nLower-error clients:"
)

print(
    best_3[
        [
            "client_id",
            "nrmse",
            "coverage",
            "nwinkler"
        ]
    ]
)


print(
    "\nHigher-error clients:"
)

print(
    worst_3[
        [
            "client_id",
            "nrmse",
            "coverage",
            "nwinkler"
        ]
    ]
)


# ============================================================
# 17. REPRESENTATIVE PLOT
# ============================================================

def plot_autoarima(
    result,
    title
):

    # Last valid test window
    actual = (

        result[
            "actual"
        ][-1]
    )


    forecast = (

        result[
            "forecast"
        ][-1]
    )


    lower = (

        result[
            "lower_conf"
        ][-1]
    )


    upper = (

        result[
            "upper_conf"
        ][-1]
    )


    history = (

        result[
            "history_values"
        ]
    )


    history_dates = (

        result[
            "history_dates"
        ]
    )


    future_dates = (

        result[
            "future_dates"
        ]
    )


    # --------------------------------------------------------
    # Window-specific metrics
    # --------------------------------------------------------

    window_rmse = np.sqrt(

        mean_squared_error(
            actual,
            forecast
        )
    )


    window_coverage = np.mean(

        (
            actual
            >= lower
        )

        &

        (
            actual
            <= upper
        )
    )


    window_winkler = winkler_score(

        actual,

        lower,

        upper,

        alpha=ALPHA
    )


    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    plt.figure(
        figsize=(15, 6)
    )


    plt.plot(

        history_dates,

        history,

        linewidth=2,

        label="History"
    )


    plt.plot(

        future_dates,

        actual,

        linewidth=2,

        label="Actual"
    )


    plt.plot(

        future_dates,

        forecast,

        linewidth=2,

        label="Forecast"
    )


    plt.fill_between(

        future_dates,

        lower,

        upper,

        alpha=0.25,

        label="90% Conformal PI"
    )


    plt.axvline(

        history_dates[-1],

        linestyle="--",

        linewidth=2,

        label="Forecast Start"
    )


    plt.title(

        f"{title}\n"

        f"Client NRMSE="
        f"{result['nrmse']:.3f} | "

        f"Client Coverage="
        f"{result['coverage']:.2%} | "

        f"Client NWinkler="
        f"{result['nwinkler']:.3f}\n"

        f"Window RMSE="
        f"{window_rmse:,.0f} EUR | "

        f"Window Coverage="
        f"{window_coverage:.2%} | "

        f"Window Winkler="
        f"{window_winkler:,.0f} EUR"
    )


    plt.xlabel(
        "Date"
    )

    plt.ylabel(
        "Balance (EUR)"
    )


    plt.xticks(
        rotation=45
    )


    plt.grid(
        alpha=0.3
    )


    plt.legend()


    plt.tight_layout()


    plt.show()


# ============================================================
# 18. PLOT LOWER-ERROR EXAMPLES
# ============================================================

for i, client_id in enumerate(

    best_3[
        "client_id"
    ],

    start=1
):

    plot_autoarima(

        all_results[
            client_id
        ],

        f"Lower-Error AutoARIMA Forecast {i}"
    )


# ============================================================
# 19. PLOT HIGHER-ERROR EXAMPLES
# ============================================================

for i, client_id in enumerate(

    worst_3[
        "client_id"
    ],

    start=1
):

    plot_autoarima(

        all_results[
            client_id
        ],

        f"Higher-Error AutoARIMA Forecast {i}"
    )


# ============================================================
# 20. SAVE RESULTS
# ============================================================

client_output_file = (
    f"autoarima_client_metrics"
    f"_pool_{POOL_NAME}"
    f"_top_{N_SELECTED_CLIENTS}.csv"
)


aggregate_output_file = (
    f"autoarima_aggregate_metrics"
    f"_pool_{POOL_NAME}"
    f"_top_{N_SELECTED_CLIENTS}.csv"
)


metrics_df.to_csv(

    client_output_file,

    index=False
)


pd.DataFrame(
    [
        aggregate_metrics
    ]
).to_csv(

    aggregate_output_file,

    index=False
)


print(
    "\nSaved:"
)

print(
    client_output_file
)

print(
    aggregate_output_file
)